GD_Landsat_03_Select picks which images to use in animations and adds a column to LandsatMetadata.csv accordingly.
Some code from GD_Landsat_02, some from prompt: "in a python notebook, load a list of image file names into a pandas dataframe. Open each image in turn and have a checkbox labeled "Keep?" that stores true if checked or false if not checked into a new column of the dataframe"
#See also: GD_Landsat_04_Animate, GD_Landsat_03a_SelectWithCloudFraction (on land or total)

Google AI says: you can use ipywidgets to create an interactive interface with an image display and a checkbox, and a pandas DataFrame to store the results. The process involves iterating through the images and using an observer function to update the DataFrame when the checkbox state changes. 
Then spits out some broken code, then fixes its code, and adds functionality...

Add logic to only add new Keep? column if it doesn't already exist. Allow rename of old Keep column to add new Keep column(s).

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import glob

In [ ]:
folder_base = r'C:\Users\andyb\Documents\U\SEAN_Glacier-Dynamics' #os.path.join()
folder_shp = r'C:\Users\andyb\Documents\U\GEE-Courses\data'
file_path=os.path.join(folder_base,'glacierPropsLandsat.csv')

glaciers = pd.read_csv(file_path) #contains Name, LatCenter, LonCenter, two types of bounding boxes (see GD_Landsat_01_Setup).
glaciers['Name']

In [ ]:
# Create a dummy image directory and some image files for demonstration
os.makedirs(os.path.join(folder_shp,"test_images"), exist_ok=True)
for i in range(8):
    img = Image.new('RGB', (100, 100), color = ('red' if i%2==0 else ('blue' if i%3==0 else 'green')))
    img.save(os.path.join(folder_shp,f'test_images/image_{i}.png'))

In [ ]:
# Load file paths into a DataFrame
image_files = glob.glob(os.path.join(folder_shp,"test_images/*.png"))
df = pd.DataFrame(image_files, columns=['ImagePath'])
df['ImageName'] = df['ImagePath'].apply(os.path.basename)

# Initialize a new 'Keep?' column with False
#df['Keep?'] = False
# Force the column to native Python booleans during initialization to avoid "TraitError: The 'value' trait of a Checkbox instance expected a boolean, not the bool np.False_."
df['Keep?'] = pd.Series([False] * len(df), dtype=object)

print("Initial DataFrame:")
display(df)

In [ ]:
#Works, only has next button 
def label_images(df):
    index = 0
    
    # UI Elements
    img_widget = widgets.Image(width=300)
    checkbox = widgets.Checkbox(description='Keep?')
    btn_next = widgets.Button(description="Next Image")
    out = widgets.Output()

    def update_ui():
        nonlocal index
        if index < len(df):
            # Load and display image
            with open(df.iloc[index]['ImagePath'], "rb") as file: #df_to_label.iloc[current_index]['ImagePath']
                img_widget.value = file.read()
            # Set checkbox to current value in DF
            checkbox.value = df.at[index, 'Keep?']
        else:
            with out:
                clear_output()
                print("Finished labeling all images!")
                display(df)

    def on_next_clicked(b):
        nonlocal index
        # Save current checkbox state to DF
        df.at[index, 'Keep?'] = checkbox.value
        index += 1
        update_ui()

    btn_next.on_click(on_next_clicked)
    
    # Display the tool
    update_ui()
    display(widgets.VBox([img_widget, checkbox, btn_next, out]))

label_images(df)


In [ ]:
#WORKS adding previous button and filename, and progress x/n
def label_images(df):
    index = 0
    
    # UI Elements
    filename_label = widgets.Label() # Displays the filename
    img_widget = widgets.Image(width=300)
    checkbox = widgets.Checkbox(description='Keep?')
    
    btn_prev = widgets.Button(description="Previous Image", icon="arrow-left")
    btn_next = widgets.Button(description="Next Image", icon="arrow-right")
    
    # Layout the buttons side-by-side
    nav_buttons = widgets.HBox([btn_prev, btn_next])
    out = widgets.Output()

    def update_ui():
        nonlocal index
        with out:
            clear_output()
            if 0 <= index < len(df):
                fname = df.iloc[index]['ImageName']
                fpath = df.iloc[index]['ImagePath'] #AKB added
                filename_label.value = f"Current File: {fname} ({index + 1}/{len(df)})"
                
                with open(fpath, "rb") as file:
                    img_widget.value = file.read()
                
                # Use bool() cast to avoid TraitError
                checkbox.value = bool(df.at[index, 'Keep?'])
            elif index >= len(df):
                print("Finished labeling! Showing results:")
                display(df)

    def on_next_clicked(b):
        nonlocal index
        if index < len(df):
            df.at[index, 'Keep?'] = checkbox.value
            index += 1
            update_ui()

    def on_prev_clicked(b):
        nonlocal index
        if index > 0:
            # Save current state before going back
            if index < len(df):
                df.at[index, 'Keep?'] = checkbox.value
            index -= 1
            update_ui()

    btn_next.on_click(on_next_clicked)
    btn_prev.on_click(on_prev_clicked)
    
    update_ui()
    # Stack everything vertically
    display(widgets.VBox([img_widget, filename_label, checkbox, nav_buttons, out]))

label_images(df)


In [ ]:
#BROKEN adding skip to index, and keyboard shortcuts (too complicated)
def label_images(df):
    index = 0
    
    # UI Elements
    filename_label = widgets.Label()
    img_widget = widgets.Image(width=300)
    checkbox = widgets.Checkbox(description='Keep? (Press "K")')
    
    btn_prev = widgets.Button(description="Prev (Left Arrow)", icon="arrow-left")
    btn_next = widgets.Button(description="Next (Right Arrow)", icon="arrow-right")
    
    # Jump to index box
    jump_box = widgets.BoundedIntText(value=0, min=0, max=len(df)-1, description='Jump to:', continuous_update=False)
    
    # Keyboard shortcut listener (Javascript bridge)
    # Left arrow: 37, Right arrow: 39, 'k' key: 75
    js_listener = widgets.HTML('''
        <script>
            document.addEventListener('keydown', function(e) {
                if (e.keyCode == 37) { // Left
                    document.querySelector('.btn-prev').click();
                } else if (e.keyCode == 39) { // Right
                    document.querySelector('.btn-next').click();
                } else if (e.keyCode == 75) { // 'k' for keep
                    document.querySelector('.chk-keep input').click();
                }
            });
        </script>
    ''')
    
    # Add classes for JS targeting
    btn_prev.add_class('btn-prev')
    btn_next.add_class('btn-next')
    checkbox.add_class('chk-keep')

    nav_buttons = widgets.HBox([btn_prev, btn_next, jump_box])
    out = widgets.Output()

    def update_ui():
        nonlocal index
        with out:
            clear_output()
            if 0 <= index < len(df):
                fname = df.iloc[index]['ImageName']
                fpath = df.iloc[index]['ImagePath'] #AKB added
                filename_label.value = f"Current File: {fname} ({index + 1}/{len(df)})"
                filename_label.value = f"Current File: {fname} ({index + 1}/{len(df)})"
                jump_box.value = index
                with open(fpath, "rb") as file:
                    img_widget.value = file.read()
                checkbox.value = bool(df.at[index, 'Keep?'])
            elif index >= len(df):
                print("Finished labeling! Results:")
                display(df)

    def on_next_clicked(b=None):
        nonlocal index
        if index < len(df):
            df.at[index, 'Keep?'] = checkbox.value
            index += 1
            update_ui()

    def on_prev_clicked(b=None):
        nonlocal index
        if index > 0:
            if index < len(df): df.at[index, 'Keep?'] = checkbox.value
            index -= 1
            update_ui()

    def on_jump(change):
        nonlocal index
        df.at[index, 'Keep?'] = checkbox.value
        index = change['new']
        update_ui()

    btn_next.on_click(on_next_clicked)
    btn_prev.on_click(on_prev_clicked)
    jump_box.observe(on_jump, names='value')
    
    update_ui()
    display(widgets.VBox([js_listener, img_widget, filename_label, checkbox, nav_buttons, out]))

label_images(df)


In [ ]:
#BROKEN adding skip to index, deleting keyboard shortcuts (too complicated)
def label_images(df):
    index = 0
    
    # UI Elements
    filename_label = widgets.Label()
    img_widget = widgets.Image(width=300)
    checkbox = widgets.Checkbox(description='Keep? (Press "K")')
    
    btn_prev = widgets.Button(description="Prev (Left Arrow)", icon="arrow-left")
    btn_next = widgets.Button(description="Next (Right Arrow)", icon="arrow-right")
    
    # Jump to index box
    jump_box = widgets.BoundedIntText(value=0, min=0, max=len(df)-1, description='Jump to:', continuous_update=False)
    
    nav_buttons = widgets.HBox([btn_prev, btn_next, jump_box])
    out = widgets.Output()

    def update_ui():
        nonlocal index
        with out:
            clear_output()
            if 0 <= index < len(df):
                fname = df.iloc[index]['ImageName']
                fpath = df.iloc[index]['ImagePath'] #AKB added
                filename_label.value = f"Current File: {fname} ({index + 1}/{len(df)})"
                filename_label.value = f"Current File: {fname} ({index + 1}/{len(df)})"
                jump_box.value = index
                with open(fpath, "rb") as file:
                    img_widget.value = file.read()
                checkbox.value = bool(df.at[index, 'Keep?'])
            elif index >= len(df):
                print("Finished labeling! Results:")
                display(df)

    def on_next_clicked(b=None):
        nonlocal index
        if index < len(df):
            df.at[index, 'Keep?'] = checkbox.value
            index += 1
            update_ui()

    def on_prev_clicked(b=None):
        nonlocal index
        if index > 0:
            if index < len(df): df.at[index, 'Keep?'] = checkbox.value
            index -= 1
            update_ui()

    def on_jump(change):
        nonlocal index
        df.at[index, 'Keep?'] = checkbox.value
        index = change['new']
        update_ui()

    btn_next.on_click(on_next_clicked)
    btn_prev.on_click(on_prev_clicked)
    jump_box.observe(on_jump, names='value')
    
    update_ui()
    display(widgets.VBox([img_widget, filename_label, checkbox, nav_buttons, out]))

label_images(df)


In [ ]:
#GOOD with jump box but no Javascript
def label_images(df):
    index = 0
    
    # UI Elements
    filename_label = widgets.Label()
    img_widget = widgets.Image(width=300)
    checkbox = widgets.Checkbox(description='Keep?')
    
    btn_prev = widgets.Button(description=" ", icon="arrow-left") #description="Previous"
    btn_next = widgets.Button(description=" ", icon="arrow-right") #description="Next"
    
    # Jump Box: constraints values between 0 and the end of the dataframe
    jump_box = widgets.BoundedIntText(
        value=0, 
        min=0, 
        max=len(df)-1, 
        description='Jump:', #'Jump to Index:',
        style={'description_width': 'initial'}
    )
    
    nav_buttons = widgets.HBox([btn_prev, btn_next, jump_box])
    out = widgets.Output()

    def update_ui():
        nonlocal index
        with out:
            clear_output()
            if 0 <= index < len(df):
                fname = df.iloc[index]['ImageName']
                fpath = df.iloc[index]['ImagePath'] #AKB added
                filename_label.value = f"{fname} ({index}/{len(df)-1})" #f"File: {fname} | Index: {index} (Total: {len(df)})"
                
                # Update jump box value without triggering the observer
                jump_box.unobserve(on_jump, names='value')
                jump_box.value = index
                jump_box.observe(on_jump, names='value')
                
                with open(fpath, "rb") as file:
                    img_widget.value = file.read()
                
                # Cast to native bool for ipywidgets compatibility
                checkbox.value = bool(df.at[index, 'Keep?'])
            else:
                print("End of list.")

    def on_next_clicked(b):
        nonlocal index
        if index < len(df):
            df.at[index, 'Keep?'] = checkbox.value
            index += 1
            if index < len(df):
                update_ui()
            else:
                with out:
                    clear_output()
                    print("Finished! Final DataFrame:")
                    display(df)

    def on_prev_clicked(b):
        nonlocal index
        if index > 0:
            df.at[index, 'Keep?'] = checkbox.value
            index -= 1
            update_ui()

    def on_jump(change):
        nonlocal index
        # Save current state before jumping
        df.at[index, 'Keep?'] = checkbox.value
        index = change['new']
        update_ui()

    # Event Handlers
    btn_next.on_click(on_next_clicked)
    btn_prev.on_click(on_prev_clicked)
    jump_box.observe(on_jump, names='value')
    
    # Initial display
    update_ui()
    display(widgets.VBox([img_widget, filename_label, checkbox, nav_buttons, out]))

In [ ]:
label_images(df)

In [ ]:
#Not working, more complicated from first attempt with google ai
def image_labeler(df_to_label):
    # Use an output widget to manage the display area
    output = widgets.Output()

    # Function to update the DataFrame when the checkbox value changes
    def on_checkbox_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            df_to_label.loc[df_to_label['ImagePath'] == current_image_path, 'Keep?'] = change['new']
            # Optional: display the updated part of the dataframe
            with output:
                clear_output(wait=True)
                display(df_to_label[df_to_label['ImagePath'] == current_image_path])

    # Create widgets
    checkbox = widgets.Checkbox(value=bool(False), description='Keep?')
    checkbox.observe(on_checkbox_change, names='value')

    next_button = widgets.Button(description="Next Image")
    prev_button = widgets.Button(description="Previous Image")

    # Image widget
    image_widget = widgets.Image(format='png', width=300, height=300)

    # Index tracker
    global current_index
    current_index = 0
    global current_image_path
    current_image_path = ""

    def update_image_display():
        global current_index, current_image_path
        if current_index >= 0 and current_index < len(df_to_label):
            current_image_path = df_to_label.iloc[current_index]['ImagePath']
            with open(current_image_path, "rb") as f:
                image_widget.value = f.read()
            # Update checkbox value to reflect current state in DataFrame
            checkbox.value = df_to_label.iloc[current_index]['Keep?']
            with output:
                clear_output(wait=True)
                print(f"Displaying image {current_index + 1}/{len(df_to_label)}: {os.path.basename(current_image_path)}")
                display(df_to_label.iloc[current_index:current_index+1])
        else:
            with output:
                clear_output(wait=True)
                print("All images processed.")
            checkbox.disabled = True
            next_button.disabled = True
            prev_button.disabled = True

    def on_next_button_clicked(_):
        global current_index
        if current_index < len(df_to_label) - 1:
            current_index += 1
            update_image_display()

    def on_prev_button_clicked(_):
        global current_index
        if current_index > 0:
            current_index -= 1
            update_image_display()

    next_button.on_click(on_next_button_clicked)
    prev_button.on_click(on_prev_button_clicked)

    # Initial display
    update_image_display()

    # Arrange and display widgets
    display(widgets.VBox([widgets.HBox([prev_button, next_button, checkbox]), image_widget, output]))

# Run the function
image_labeler(df)


In [ ]:
#choose one glacier 
glacier = glaciers.iloc[0] #0=Margerie, 12=McBride
glacierdf=glaciers.iloc[[0]]
print('You chose: ' + glacier['Name'])
folder_out=os.path.join(folder_shp, glacier['Name'])
folder_fig=os.path.join(folder_out, 'Figures')

In [ ]:
# load metadata from CSV
file_meta = os.path.join(folder_out, 'LandsatMetadata.csv')
mdf = pd.read_csv(file_meta, parse_dates=['DATE_ACQUIRED', 'datetime'])
print(f"Image info loaded from: {file_meta}")

In [ ]:
#type(mdf) #pandas.core.frame.DataFrame
print(mdf.dtypes)
print(mdf.columns) #Index(['system:id', 'system:index', 'DATE_ACQUIRED', 'system:time_start', 'CLOUD_COVER', 'CLOUD_COVER_LAND', 'datetime', 'datetimestr'],      dtype='object')

In [ ]:
print(mdf['datetime'][0])
type(mdf['datetime'][0]) #pandas._libs.tslibs.timestamps.Timestamp #was str then I added parse_dates to the read_csv line

In [ ]:
# Count points per year
mdf['year'] = mdf['datetime'].dt.year
points_per_year = mdf['year'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 6))
# Bar chart
bars = ax.bar(points_per_year.index, points_per_year.values,
              color='steelblue', edgecolor='black', alpha=0.8)
# Add value labels on top of bars
#for bar in bars:
#    height = bar.get_height()
#    ax.text(bar.get_x() + bar.get_width()/2, height + 0.5,
#            f'{int(height)}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Year')
ax.set_ylabel('Images')
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
ax.set_xticks(points_per_year.index) # Optional: force integer x-ticks
ax.tick_params(axis='x', rotation=90) #better(?) than plt.xticks(rotation=90)
#fig.savefig(os.path.join(folder_fig,glacier['Name']+'CountPerYear.png'), dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()